# Preprocessing Pipeline
Cleaning up the raw data before modeling.

TRAIN.csv -> TRAIN_PREPROCESSED.csv
TEST.csv -> TEST_PREPROCESSED.csv

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import PowerTransformer, StandardScaler
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

## Load data

In [ ]:
df_raw = pd.read_csv('TRAIN.csv')
print(f'Shape: {df_raw.shape}')
print(f'Missing: {df_raw.isnull().sum().sum()}')
print(f'Duplicates: {df_raw.duplicated().sum()}')
print(f'\nClass counts:')
print(df_raw['Class'].value_counts().to_string())
df_raw.head(3)

## Remove duplicates
738 duplicates found - dropping them

In [ ]:
before = len(df_raw)
df = df_raw.drop_duplicates().reset_index(drop=True)
after = len(df)

print(f'Removed {before - after} duplicates')
print(f'{before} -> {after} rows')

counts = df['Class'].value_counts()
for cls in sorted(counts.index):
    print(f'Class {cls}: {counts[cls]} ({counts[cls]/len(df)*100:.1f}%)')

## Split features and target

In [ ]:
features = [c for c in df.columns if c != 'Class']
X = df[features].copy()
y = df['Class'].copy()
X_original = X.copy()  # keeping a copy to compare later

print(f'X: {X.shape}')
print(f'y: {y.shape}')
X.describe().T[['mean', 'std', 'min', 'max']].round(4).head(10)

## Outlier capping
Capping at 1st and 99th percentiles instead of removing rows

In [ ]:
outlier_report = []

for feat in features:
    p1 = X[feat].quantile(0.01)
    p99 = X[feat].quantile(0.99)
    
    n_low = (X[feat] < p1).sum()
    n_high = (X[feat] > p99).sum()
    total = n_low + n_high
    
    if total > 0:
        outlier_report.append({'Feature': feat, 'Capped': total,
                              'Pct': round(total/len(X)*100, 2)})
    
    X[feat] = X[feat].clip(lower=p1, upper=p99)

outlier_df = pd.DataFrame(outlier_report)
print(f'Features capped: {len(outlier_df)}/{len(features)}')
print(f'Total values capped: {outlier_df["Capped"].sum()}')
outlier_df.sort_values('Capped', ascending=False).head(10)

## Fix skewness with Yeo-Johnson
44 out of 47 features are heavily skewed - need to fix this

In [ ]:
skew_before = X.skew()
n_skewed_before = (skew_before.abs() > 1).sum()
print(f'Before: {n_skewed_before} highly skewed features')

pt = PowerTransformer(method='yeo-johnson', standardize=False)
X_transformed = pd.DataFrame(pt.fit_transform(X), columns=features, index=X.index)

skew_after = X_transformed.skew()
n_skewed_after = (skew_after.abs() > 1).sum()
print(f'After: {n_skewed_after} highly skewed features')
print(f'Fixed {n_skewed_before - n_skewed_after} features')

X = X_transformed.copy()

In [ ]:
# quick visual comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

sorted_before = skew_before.sort_values(key=abs, ascending=False)
colors_b = ['red' if abs(v) > 1 else 'green' for v in sorted_before]
axes[0].barh(sorted_before.index, sorted_before.values, color=colors_b, alpha=0.6)
axes[0].axvline(0, color='black')
axes[0].axvline(-1, color='gray', linestyle='--', alpha=0.5)
axes[0].axvline(1, color='gray', linestyle='--', alpha=0.5)
axes[0].set_title('BEFORE')
axes[0].invert_yaxis()

sorted_after = skew_after.reindex(sorted_before.index)
colors_a = ['red' if abs(v) > 1 else 'green' for v in sorted_after]
axes[1].barh(sorted_after.index, sorted_after.values, color=colors_a, alpha=0.6)
axes[1].axvline(0, color='black')
axes[1].axvline(-1, color='gray', linestyle='--', alpha=0.5)
axes[1].axvline(1, color='gray', linestyle='--', alpha=0.5)
axes[1].set_title('AFTER (Yeo-Johnson)')
axes[1].invert_yaxis()

plt.suptitle('Skewness Before vs After', fontweight='bold')
plt.tight_layout()
plt.show()

## Remove highly correlated features
Some features are basically copies of each other (|r| > 0.9). Dropping the one that's less useful for predicting the target.

In [ ]:
corr_matrix = X.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

high_corr_pairs = []
for col in upper.columns:
    for idx in upper.index:
        if upper.loc[idx, col] > 0.9:
            high_corr_pairs.append((idx, col, upper.loc[idx, col]))

print(f'Found {len(high_corr_pairs)} highly correlated pairs\n')

# figure out which one to drop from each pair
target_corr = {feat: abs(X[feat].corr(y)) for feat in features if feat in X.columns}
features_to_drop = set()

for f1, f2, r in high_corr_pairs:
    c1 = target_corr.get(f1, 0)
    c2 = target_corr.get(f2, 0)
    drop = f2 if c1 >= c2 else f1
    features_to_drop.add(drop)
    print(f'{f1} <-> {f2} (r={r:.3f}) -> drop {drop}')

print(f'\nDropping {len(features_to_drop)} features')
X = X.drop(columns=list(features_to_drop))
print(f'Shape now: {X.shape}')

## Scale features
StandardScaler to normalize everything to mean=0, std=1

In [ ]:
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)

print(f'Scaled {X_scaled.shape[1]} features')
print(f'Mean near 0: {(X_scaled.mean().abs() < 0.001).all()}')
print(f'Std near 1: {((X_scaled.std() - 1).abs() < 0.01).all()}')

X = X_scaled.copy()

## Before vs After comparison

In [ ]:
show_feats = [f for f in ['F01', 'F09', 'F19', 'F05', 'F07', 'F30'] if f in X.columns]

fig, axes = plt.subplots(len(show_feats), 2, figsize=(14, len(show_feats) * 2.5))

for i, feat in enumerate(show_feats):
    axes[i, 0].hist(X_original[feat], bins=50, color='steelblue', alpha=0.7)
    axes[i, 0].set_title(f'{feat} - Raw')
    axes[i, 0].text(0.95, 0.85, f'skew={X_original[feat].skew():.2f}',
                    transform=axes[i, 0].transAxes, ha='right', fontsize=9)
    
    axes[i, 1].hist(X[feat], bins=50, color='green', alpha=0.7)
    axes[i, 1].set_title(f'{feat} - Cleaned')
    axes[i, 1].text(0.95, 0.85, f'skew={X[feat].skew():.2f}',
                    transform=axes[i, 1].transAxes, ha='right', fontsize=9)

plt.suptitle('Before vs After', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# final correlation heatmap
fig, ax = plt.subplots(figsize=(14, 11))
corr_final = X.corr()
mask = np.triu(np.ones_like(corr_final, dtype=bool))
sns.heatmap(corr_final, mask=mask, cmap='coolwarm', center=0, annot=False, ax=ax)
ax.set_title('Correlation After Preprocessing')
plt.tight_layout()
plt.show()

max_corr = corr_final.abs().where(np.triu(np.ones(corr_final.shape), k=1).astype(bool)).max().max()
print(f'Max correlation remaining: {max_corr:.4f}')

## Save cleaned data
Applying same preprocessing to TEST.csv too

In [ ]:
# save train
df_preprocessed = X.copy()
df_preprocessed['Class'] = y.values
df_preprocessed.to_csv('TRAIN_PREPROCESSED.csv', index=False)
print(f'Saved TRAIN_PREPROCESSED.csv: {df_preprocessed.shape}')

# preprocess test data with same steps
df_test = pd.read_csv('TEST.csv')
test_id = df_test['ID'] if 'ID' in df_test.columns else None
test_features = [c for c in df_test.columns if c not in ['ID', 'Class']]
X_test = df_test[test_features].copy()

# outlier capping using train percentiles
for feat in test_features:
    p1 = X_original[feat].quantile(0.01) if feat in X_original.columns else X_test[feat].quantile(0.01)
    p99 = X_original[feat].quantile(0.99) if feat in X_original.columns else X_test[feat].quantile(0.99)
    X_test[feat] = X_test[feat].clip(lower=p1, upper=p99)

# yeo-johnson
X_test_transformed = pd.DataFrame(pt.transform(X_test[features]), columns=features, index=X_test.index)
X_test_transformed = X_test_transformed.drop(columns=list(features_to_drop))

# scale
X_test_scaled = pd.DataFrame(scaler.transform(X_test_transformed), columns=X_test_transformed.columns, index=X_test.index)

df_test_preprocessed = X_test_scaled.copy()
if test_id is not None:
    df_test_preprocessed.insert(0, 'ID', test_id.values)

df_test_preprocessed.to_csv('TEST_PREPROCESSED.csv', index=False)
print(f'Saved TEST_PREPROCESSED.csv: {df_test_preprocessed.shape}')

## Summary

In [ ]:
print('Preprocessing done!')
print(f'Raw: {df_raw.shape[0]} rows, {df_raw.shape[1]} cols')
print(f'After dedup: {after} rows')
print(f'Features: {len(features)} -> {X.shape[1]} (dropped {len(features_to_drop)} correlated)')
print(f'Skewed features fixed: {n_skewed_before} -> {n_skewed_after}')
print(f'\nSaved: TRAIN_PREPROCESSED.csv, TEST_PREPROCESSED.csv')